# G15 - truncation or shrinkage? The two disagree

G14 gave a contradiction my pre-registration hid: heavy ridge recovers the cliff (0.009 -> 0.902) while the minimum-norm head does nothing (0.009). If the cause were a removable set of directions, truncation should be the CLEANEST repair - it is not.

**First clear the confound:** G14's rcond only reached 1e-4 and may have truncated nothing. This sweeps EXPLICIT rank k, including 574 (the effective rank) and 768 (the ambient dimension, where the cliff is), so the amount of truncation is known rather than implied.

If some k works, the k that works settles which boundary matters. If none does while shrinkage succeeds, the cause is not a subspace at all.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G15 — truncation or shrinkage? The two disagree, and that is the clue.
# Run after G14. Requires the same caches; rebuilds the hub itself.
#
# WHERE THIS COMES FROM, INCLUDING A DESIGN ERROR OF MINE. G14 tested a
# conditioning account of the width cliff two ways and pre-registered
# "either suffices". That was wrong, and the results show why:
#
#   heavy ridge on the head   alpha 1e2   transfer 0.009 -> 0.902
#   minimum-norm (pinv)       any rcond   transfer 0.009 -> 0.009
#
# If conditioning were the mechanism, the minimum-norm head should be the
# CLEANEST repair available - it places exactly zero weight outside the
# source's row space. It does nothing. Heavy shrinkage, which damps every
# direction including well-spanned ones, recovers almost everything. A
# gate that accepts "either" hid a contradiction instead of surfacing it.
#
# THE DISTINCTION THAT MATTERS. Ridge SHRINKS: every direction is damped
# by lambda/(lambda+alpha). Pseudo-inverse TRUNCATES: directions below a
# cutoff are removed entirely, the rest untouched. The G14 data says
# shrinkage repairs the cliff and truncation does not. If the problem were
# a set of junk directions to be deleted, truncation would be the better
# tool - so the "junk directions" reading is in trouble.
#
# ONE CONFOUND TO CLEAR FIRST. G14's rcond only reached 1e-4, and on a
# smoothly decaying spectrum that may truncate almost nothing. B might
# have failed by never having truncated at all. This sweeps EXPLICIT rank
# k rather than rcond, so the amount of truncation is known rather than
# implied.
#
# THE TEST. Fit the head with exactly k singular values retained, for k
# spanning below the effective rank, at it, at the ambient dimension, and
# at the full width. Then compare against the shrinkage curve on the same
# axes.
#
#   TRUNCATION WORKS at some k -> the junk-direction reading survives in a
#     sharper form, and the right k tells us which boundary matters: the
#     effective rank (574) or the ambient dimension (768). Those two
#     numbers have been in tension since G13 and this would settle it.
#
#   TRUNCATION NEVER WORKS while shrinkage does -> the cliff is not caused
#     by a removable set of directions. Something about damping ALL
#     directions is what helps, which points at scale mismatch between the
#     source's hub coordinates and the test encoders' rather than at any
#     subspace. That is a different mechanism and would need its own test.
#
# PRE-REGISTERED: truncation passes if some k recovers transfer above
# 0.802 (within 15 points of the below-wall 0.952). Partial recovery is
# not a pass. If it passes, the k at which it does is the finding.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ENTRY_ALPHA, N_EVAL, SEED = 1e-2, 1000, 0
SOURCE = "img_small"
BASE_WIDTH, PROBE_WIDTH = 512, 1024
EFF_RANK = 574                     # measured in G13
RECOVER = 0.802                    # 0.952 - 0.15
KS = [128, 256, 400, 512, 574, 640, 700, 768, 800, 896, 1024]
ALPHAS = [1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3, 1e4]

SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]
SOURCES = ["img_small", "img_base", "img_large"]
D_SRC = SPACES[SOURCE].shape[1]

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]
T = SPACES["txt_bge"]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
H = (_ref - _mu) @ (_VT[:PROBE_WIDTH].T /
                    (_sv[:PROBE_WIDTH] / np.sqrt(len(_ref))))
TO_HUB = {k: ridge(SPACES[k][tr], H, ENTRY_ALPHA) for k in SOURCES}
C = SPACES[SOURCE][tr] @ TO_HUB[SOURCE]
GAL = l2n(T[te])

# the source's own spectrum inside the hub, so the k grid can be read
Uc, Sc, Vtc = np.linalg.svd(C - C.mean(0), full_matrices=False)
tot = (Sc ** 2).sum()
print(f"source hub coordinates {C.shape}; singular spectrum:")
for k in (128, 400, 574, 768, 1024):
    if k <= len(Sc):
        print(f"  top {k:5d} directions hold "
              f"{(Sc[:k] ** 2).sum() / tot:6.2%} of the variance"
              f"   sigma_k = {Sc[k-1]:.4g}")


C_MU = C.mean(0)
T_MU = T[tr].mean(0)


def evaluate(head, centred=False):
    """centred=True for the truncated heads, which were fitted on centred
    C and centred T and therefore need the same treatment at eval time."""
    out = []
    for enc in SOURCES:
        if enc == SOURCE:
            continue
        Ce = SPACES[enc][te] @ TO_HUB[enc]
        pred = ((Ce - C_MU) @ head + T_MU) if centred else (Ce @ head)
        zero = r1(pred, GAL)
        nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr], ENTRY_ALPHA), GAL)
        out.append(zero / max(nat, 1e-9))
    return float(np.mean(out))


def head_truncated(k):
    """Least squares keeping EXACTLY k singular directions of C.

    Explicit rank, so the amount of truncation is known - unlike rcond,
    which depends on the spectrum and may truncate nothing at all. This is
    what G14's minimum-norm test should have swept.

    The SVD is taken on centred C, so the same centring is applied when
    the head is used; the constant term is folded in by predicting the
    centred target and adding its mean back.
    """
    Uk, Sk, Vk = Uc[:, :k], Sc[:k], Vtc[:k]
    return Vk.T @ np.diag(1.0 / Sk) @ Uk.T @ (T[tr] - T[tr].mean(0))


def head_ridge(a):
    return ridge(C, T[tr], a)


print("\n" + "=" * 66)
print("TRUNCATION - exactly k directions retained")
print("=" * 66)
print(f"{'k':>7}{'transfer':>11}{'var kept':>11}   note")
trunc = {}
for k in KS:
    if k > min(C.shape):
        continue
    v = evaluate(head_truncated(k), centred=True)
    trunc[k] = v
    note = ""
    if k == EFF_RANK:
        note = "<- effective rank (G13)"
    elif k == D_SRC:
        note = "<- ambient dimension, where the cliff is"
    elif k == PROBE_WIDTH:
        note = "<- full width, equals plain least squares"
    print(f"{k:>7}{v:>11.3f}{(Sc[:k] ** 2).sum() / tot:>11.2%}   {note}")

print("\n" + "=" * 66)
print("SHRINKAGE - ridge, every direction damped")
print("=" * 66)
print(f"{'alpha':>10}{'transfer':>11}")
shrink = {}
for a in ALPHAS:
    v = evaluate(head_ridge(a))
    shrink[a] = v
    print(f"{a:>10.0e}{v:>11.3f}")

In [ ]:
# ---------- read it ----------
best_k = max(trunc, key=trunc.get)
best_a = max(shrink, key=shrink.get)
t_pass = trunc[best_k] > RECOVER
s_pass = shrink[best_a] > RECOVER

print("\n" + "=" * 66)
print(f"  best truncation   k = {best_k:5d}   {trunc[best_k]:.3f}")
print(f"  best shrinkage    alpha = {best_a:.0e}   {shrink[best_a]:.3f}")
print(f"  pass threshold                  {RECOVER:.3f}")
print()
if t_pass:
    near_eff = abs(best_k - EFF_RANK) <= 70
    near_amb = abs(best_k - D_SRC) <= 70
    print("TRUNCATION WORKS. The cliff is caused by a removable set of")
    print("directions after all, and G14's minimum-norm result failed")
    print("because rcond never truncated enough, not because truncation is")
    print("the wrong tool.")
    if near_eff and not near_amb:
        print(f"\nThe boundary is the EFFECTIVE RANK ({EFF_RANK}), not the")
        print(f"ambient dimension ({D_SRC}). That is a problem for the")
        print("location result: the cliff is observed at the ambient")
        print("dimension, so the repair boundary and the break boundary")
        print("differ and something still has to reconcile them.")
    elif near_amb and not near_eff:
        print(f"\nThe boundary is the AMBIENT DIMENSION ({D_SRC}), matching")
        print("where the cliff is observed. That reconciles the tension G13")
        print("opened, and the rank account returns in a sharper form:")
        print("what matters is not how many directions carry variance but")
        print("how many the source could in principle produce.")
    else:
        print(f"\nThe best k ({best_k}) is near neither {EFF_RANK} nor")
        print(f"{D_SRC}. Report the curve as measured - a repair boundary")
        print("unrelated to either candidate is a finding, not a footnote.")
elif s_pass:
    print("TRUNCATION FAILS WHERE SHRINKAGE WORKS. This is the informative")
    print("outcome. No set of directions can be deleted to repair the cliff,")
    print("but damping ALL of them repairs it - so the cause is not a")
    print("subspace the head should not be reading.")
    print()
    print("What damping does that deletion does not: it reduces the head's")
    print("sensitivity to the SCALE of its input. At large alpha the fit")
    print("approaches a scaled correlation readout rather than a")
    print("least-squares inverse, and such readouts tolerate a shift")
    print("between the distribution the head was fitted on and the one it")
    print("is applied to. That points at a scale or covariance mismatch")
    print("between the source's hub coordinates and the test encoders',")
    print("not at any set of bad directions.")
    print()
    print("That is a NEW hypothesis, not a conclusion. The test it implies:")
    print("standardise each encoder's hub coordinates before the head sees")
    print("them, and see whether the cliff disappears at alpha 1e-2.")
else:
    print("NEITHER REPAIRS THE CLIFF at the pre-registered bar. Whatever")
    print("G14's alpha=1e2 result was, it does not survive this grid, and")
    print("the cliff is not a head-fitting problem at all.")

print("\nTwo things hold regardless of the above, and both belong in the")
print("report. The cliff IS repairable - 0.009 to 0.902 by changing one")
print("hyperparameter - so it is not an information-theoretic limit. And")
print("C.13's 'alpha cannot substitute for width' was measured on the")
print("ENTRY MAP alpha; the head's alpha was never swept, and it is the")
print("head that breaks. That claim needs narrowing to what was tested.")